In [ ]:
--mart_faa_stats

WITH departures AS ( 
	SELECT origin AS faa
			,COUNT(DISTINCT dest) AS nunique_to
			,COUNT(sched_dep_time) AS dep_planned
			,SUM(cancelled) AS dep_cancelled
			,SUM(diverted) AS dep_diverted
			,COUNT(arr_time) AS dep_n_flights
			-- ,COUNT(DISTINCT tail_number) AS dep_nunique_tails -- BONUS TASK
			-- ,COUNT(DISTINCT airline) AS dep_nunique_airlines -- BONUS TASK
	FROM {{ref('prep_flights')}} 
	GROUP BY origin
	ORDER BY origin
),
arrivals AS (
	SELECT dest AS faa
			,COUNT(DISTINCT origin) AS nunique_from
			,COUNT(sched_dep_time) AS arr_planned
			,SUM(cancelled) AS arr_cancelled
			,SUM(diverted) AS arr_diverted
			,COUNT(arr_time) AS arr_n_flights
			-- ,COUNT(DISTINCT tail_number) AS arr_nunique_tails -- BONUS TASK
			-- ,COUNT(DISTINCT airline) AS arr_nunique_airlines -- BONUS TASK
	FROM {{ref('prep_flights')}} 
	GROUP BY dest
	ORDER BY dest
),
total_stats AS (
	SELECT faa
			,nunique_to
			,nunique_from
	--		,(nunique_to + nunique_from)::NUMERIC/2 AS n_connections -- fractions would indicate that for the number of connections to!=from 
			,dep_planned + arr_planned AS total_planned
			,dep_cancelled + arr_cancelled AS total_cancelled
			,dep_diverted + arr_diverted AS total_diverted
	--		,((dep_cancelled + arr_cancelled + dep_diverted + arr_diverted)::NUMERIC/(dep_planned + arr_planned)::NUMERIC)*100 AS percent_change -- BONUS TASK
			,dep_n_flights + arr_n_flights AS total_flights
	--		,(dep_nunique_tails + arr_nunique_tails)::NUMERIC/2 AS nunique_tails -- fractions would indicate that for the number of tails to!=from -- BONUS TASK
	--		,(dep_nunique_airlines + arr_nunique_airlines)::NUMERIC/2 AS nunique_airlines -- fractions would indicate that for the number of airlines to!=from -- BONUS TASK
	FROM departures
	JOIN arrivals
	USING (faa)
)
SELECT a.city
		,a.country
		,a.name
		, t.* 
FROM total_stats t
LEFT JOIN {{ref('prep_airports')}} a
USING (faa)
ORDER BY total_diverted DESC

In [ ]:
--mart route stats
WITH flights_stats AS (
	SELECT -- TO_CHAR(flight_date, 'YYYY-MM') AS flight_month, --for the alternative GROUPBY also by month (see below)
		   origin
		   ,dest
        --    ,origin || ' - ' || dest AS route -- optional (challenge: keep only 'route' in the final query output, w/o 'origin' and 'dest')
		   ,COUNT(flight_number) AS n_flights
		   ,COUNT(DISTINCT tail_number) AS nunique_tails
		   ,COUNT(DISTINCT airline) AS nunique_airlines
		   ,AVG(actual_elapsed_time)::INTEGER * ('1 second'::INTERVAL) AS avg_actual_elapsed_time 
        --    ,(AVG(actual_elapsed_time)*60)::INTEGER * ('1 second'::INTERVAL) AS avg_actual_elapsed_time -- without rounding the seconds
		--    ,STDDEV(actual_elapsed_time)::INTEGER * ('1 second'::INTERVAL) AS sd_actual_elapsed_time 
		   ,AVG(arr_delay)::INTEGER * ('1 second'::INTERVAL) AS avg_arr_delay
		   ,MIN(arr_delay_interval) AS min_arr_delay
		   ,MAX(arr_delay_interval) AS max_arr_delay
		   ,SUM(cancelled) AS total_cancelled
		   ,SUM(diverted) AS total_diverted
	FROM {{ref('prep_flights')}}
    GROUP BY (origin, dest) -- over all time
	-- GROUP BY (flight_month, origin, dest) -- alternative GROUPBY also by month
),
add_names AS (
	SELECT o.city AS origin_city
			,d.city AS dest_city
			,o.name AS origin_name
			,d.name AS dest_name
			,f.*
	FROM flights_stats f
	LEFT JOIN {{ref('prep_airports')}} o
		ON origin=o.faa
	LEFT JOIN {{ref('prep_airports')}} d
		ON dest=d.faa
)
SELECT *
FROM add_names
ORDER BY (origin, dest) DESC

In [ ]:
-- mart selected faa stats weather daily

WITH departures AS ( 
	SELECT flight_date 
			,origin AS faa
			,COUNT(DISTINCT dest) AS nunique_to
			,COUNT(sched_dep_time) AS dep_planned
			,SUM(cancelled) AS dep_cancelled
			,SUM(diverted) AS dep_diverted
			,COUNT(arr_time) AS dep_n_flights
--			,COUNT(DISTINCT tail_number) AS dep_nunique_tails -- BONUS TASK
--			,COUNT(DISTINCT airline) AS dep_nunique_airlines -- BONUS TASK
	FROM {{ref('prep_flights')}}
	WHERE origin IN (SELECT DISTINCT airport_code FROM {{ref('prep_weather_daily')}})
	GROUP BY origin, flight_date 
	ORDER BY origin, flight_date
),
arrivals AS (
	SELECT flight_date
			,dest AS faa
			,COUNT(DISTINCT origin) AS nunique_from
			,COUNT(sched_dep_time) AS arr_planned
			,SUM(cancelled) AS arr_cancelled
			,SUM(diverted) AS arr_diverted
			,COUNT(arr_time) AS arr_n_flights
--			,COUNT(DISTINCT tail_number) AS arr_nunique_tails -- BONUS TASK
--			,COUNT(DISTINCT airline) AS arr_nunique_airlines -- BONUS TASK
	FROM {{ref('prep_flights')}}
	WHERE dest IN (SELECT DISTINCT airport_code FROM {{ref('prep_weather_daily')}})
	GROUP BY dest, flight_date
	ORDER BY dest, flight_date
),
total_stats AS (
	SELECT flight_date
			,faa
			,nunique_to
			,nunique_from
	--		,(nunique_to + nunique_from)::NUMERIC/2 AS n_connections -- fractions would indicate that for the number of connections to!=from --BONUS task
			,dep_planned + arr_planned AS total_planned
			,dep_cancelled + arr_cancelled AS total_cancelled
			,dep_diverted + arr_diverted AS total_diverted
	--		,((dep_cancelled + arr_cancelled + dep_diverted + arr_diverted)::NUMERIC/(dep_planned + arr_planned)::NUMERIC)*100 AS percent_change -- BONUS TASK
			,dep_n_flights + arr_n_flights AS total_flights
	--		,(dep_nunique_tails + arr_nunique_tails)::NUMERIC/2 AS nunique_tails -- fractions would indicate that for the number of tails to!=from -- BONUS TASK
	--		,(dep_nunique_airlines + arr_nunique_airlines)::NUMERIC/2 AS nunique_airlines -- fractions would indicate that for the number of airlines to!=from -- BONUS TASK
	FROM departures
	JOIN arrivals
	USING (flight_date, faa)
)
SELECT t.* 
		,w.min_temp_c
		,w.max_temp_c
		,w.precipitation_mm
		,w.max_snow_mm
		,w.avg_wind_direction
		,w.avg_wind_speed_kmh
		,w.wind_peakgust_kmh
FROM total_stats t
LEFT JOIN {{ref('prep_weather_daily')}} w
ON faa = airport_code AND flight_date = date
ORDER BY total_diverted DESC

In [ ]:
-- mart weather weekly

WITH daily_data AS (
        SELECT * 
        FROM {{ref('prep_weather_daily')}}
),
weekly_aggregation AS (
        SELECT airport_code, station_id, date_year, cw
            ,AVG(avg_temp_c)::NUMERIC(4,2) AS avg_temp_c
            ,MIN(min_temp_c)::NUMERIC(4,2) AS min_temp_c
            ,MAX(max_temp_c)::NUMERIC(4,2) AS max_temp_c
            ,SUM(precipitation_mm) AS total_prec_mm
            ,SUM(max_snow_mm) AS total_max_snow_mm
            ,AVG(avg_wind_direction)::NUMERIC(5,2) AS avg_wind_direction
            ,AVG(avg_wind_speed_kmh)::NUMERIC(5,2) AS avg_wind_speed_kmh
            ,MAX(wind_peakgust_kmh)::NUMERIC(5,2) AS wind_peakgust_kmh
            ,AVG(avg_pressure_hpa)::NUMERIC(6,2) AS avg_pressure_hpa
            ,SUM(sun_minutes) AS total_sun_minutes
            ,MODE() WITHIN GROUP (ORDER BY date_month) AS month
            ,MODE() WITHIN GROUP (ORDER BY month_name) AS month_name
            ,MODE() WITHIN GROUP (ORDER BY season) AS season
        FROM daily_data
        GROUP BY airport_code, station_id, date_year, cw
)
SELECT * FROM weekly_aggregation